# ROI spatial plots and cell-type composition

This notebook plots cell types and their respective cell masks for the specific regions of interest. Likewise, it includes visualisation of ROI on the whole section scatter plot.


## Shared setup

Imports used across the four original ROI notebooks are consolidated here. Individual figure sections retain their own figure/output settings where these differ.


In [ ]:
import h5py
import warnings
import os
import spatialdata_plot
from pathlib import Path
import logging
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties


## BE ROI morphology zooms and composition

This section comes from the BE-focused ROI composition notebook. It defines same-sized BE ROIs,
maps annotated cell boundaries onto Xenium morphology images with scale bars, and generates the
associated ROI cell-type composition plots. Repeated sample-specific plotting steps are left intact
but described once here.


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Set `sc.settings.figdir` for the following analysis.
sc.settings.figdir = "Figures_forpaper/new_colours/noWT3spatial/ROIzoomin/samesized"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Compute and store `out_dir`.
out_dir = Path("Figures_forpaper/new_colours/noWT3spatial/ROIzoomin/samesized")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad")


In [ ]:
# Define the values used for `roi_centers`.
roi_centers = {
    "BE_rep1": (1032, 3520, 100000), #little lymphoid cluster in the middle but not perfect segmentation #ROI2 #size 50 000
    "BE_rep2":  (3070, 2347, 100000), #ROI1, 300 000
    "BE_rep3": (3694, 1151, 100000), #The good one #size 150 000
    "BE_rep4":  (1814, 2032, 100000), #ROI3 #100 000
}


In [ ]:
# Define the values used for `roi_squares`.
roi_squares = {}

# Repeat the following operation for each item in the selected collection.
for sample, (x_center, y_center, roi_area_um2) in roi_centers.items():

    roi_side_um = np.sqrt(roi_area_um2)
    half = roi_side_um / 2

    roi_squares[sample] = {
        "x_center": x_center,
        "y_center": y_center,
        "x_min": x_center - half,
        "x_max": x_center + half,
        "y_min": y_center - half,
        "y_max": y_center + half,
        "side_um": roi_side_um,
        "area_um2": roi_area_um2,
    }

# Create a DataFrame for downstream analysis.
roi_squares_df = pd.DataFrame.from_dict(roi_squares, orient="index")
roi_squares_df


In [ ]:
# Select the required subset and store it as `adata.obs['x']`.
adata.obs["x"] = adata.obsm["spatial"][:, 0]
# Select the required subset and store it as `adata.obs['y']`.
adata.obs["y"] = adata.obsm["spatial"][:, 1]


In [ ]:
# Set `adata.obs['in_roi']` for the following analysis.
adata.obs["in_roi"] = False
# Set `adata.obs['roi_id']` for the following analysis.
adata.obs["roi_id"] = np.nan
# Set `adata.obs['roi_area_um2']` for the following analysis.
adata.obs["roi_area_um2"] = np.nan

# Repeat the following operation for each item in the selected collection.
for sample, roi in roi_squares_df.iterrows():

    mask = (
        (adata.obs["name"] == sample) &
        (adata.obs["x"] >= roi["x_min"]) &
        (adata.obs["x"] <= roi["x_max"]) &
        (adata.obs["y"] >= roi["y_min"]) &
        (adata.obs["y"] <= roi["y_max"])
    )

    adata.obs.loc[mask, "in_roi"] = True
    adata.obs.loc[mask, "roi_id"] = sample
    adata.obs.loc[mask, "roi_area_um2"] = roi["area_um2"]


In [ ]:
# Make an independent copy of the selected data.
adata_roi = adata[adata.obs["in_roi"]].copy()


In [ ]:
# Inspect the number of observations in each category.
adata_roi.obs['name'].value_counts()


In [ ]:
# Reset the DataFrame index after reshaping or filtering.
celltype_counts_sample = (
    adata_roi.obs
    .groupby(["name", "celltype4"])
    .size()
    .reset_index(name="n_cells")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head(20)


In [ ]:
# Calculate `celltype_counts_sample['fraction']` from the existing values.
celltype_counts_sample["fraction"] = (
    celltype_counts_sample["n_cells"] /
    celltype_counts_sample.groupby("name")["n_cells"].transform("sum")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define `plot_roi_dapi()` for reuse in the analysis below.
def plot_roi_dapi(
    sdata,
    cell_shapes_roi_px,
    celltype_colors_plot,
    sample,
    out_dir,
    x0_px, x1_px, y0_px, y1_px,
    scale="scale2",
    channel="DAPI",
    overlay_mode="boundary",   # "boundary", "filled", or "none"
    nuclei_display="gray",     # "gray" or "blue"
    fill_alpha=0.5,
    boundary_linewidth=0.4,
    filled_edgecolor="#777777",
    filled_linewidth=0.15,
    show_legend=True,
    figsize=(6, 6),
    dpi=300,

    # scale bar options
    show_scale_bar=True,
    scale_bar_um=50,
    um_per_coord=0.2125,
    scale_bar_color="white",
    scale_bar_linewidth=3,
    scale_bar_fontsize=8,
    scale_bar_location="lower right",

    # saving option
    save_label=None
):
    """
    Plot a single ROI for the DAPI channel with flexible cell overlay options.

    Important:
    This assumes x0_px/x1_px/y0_px/y1_px and cell_shapes_roi_px are in
    scale0 pixel coordinates, matching the x/y coordinates of the SpatialData image.
    """

    # get image object
    img_xr = sdata.images["morphology_focus"][scale]["image"]

    # copy and assign colors
    cell_shapes_roi_px = cell_shapes_roi_px.copy()
    cell_shapes_roi_px["plot_color"] = cell_shapes_roi_px["celltype_plot"].map(celltype_colors_plot)

    # crop DAPI image
    img_crop = img_xr.sel(
        c=channel,
        x=slice(x0_px, x1_px),
        y=slice(y0_px, y1_px)
    ).data.compute()

    # contrast scaling
    vmin, vmax = np.percentile(img_crop, [2, 99.5])
    img_norm = np.clip((img_crop - vmin) / (vmax - vmin + 1e-8), 0, 1)

    fig, ax = plt.subplots(figsize=figsize)

    # plot nuclei channel
    if nuclei_display == "gray":
        ax.imshow(
            img_crop,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    elif nuclei_display == "blue":
        rgb = np.zeros((*img_norm.shape, 3), dtype=float)
        rgb[..., 2] = img_norm
        ax.imshow(
            rgb,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    else:
        raise ValueError("nuclei_display must be 'gray' or 'blue'")

    # overlay polygons
    present_celltypes = [
        ct for ct in celltype_colors_plot
        if ct in cell_shapes_roi_px["celltype_plot"].values
    ]

    if overlay_mode == "boundary":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.boundary.plot(
                    ax=ax,
                    edgecolor=celltype_colors_plot[ct],
                    linewidth=boundary_linewidth
                )

    elif overlay_mode == "filled":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.plot(
                    ax=ax,
                    color=celltype_colors_plot[ct],
                    edgecolor=filled_edgecolor,
                    linewidth=filled_linewidth,
                    alpha=fill_alpha
                )

    elif overlay_mode == "none":
        pass

    else:
        raise ValueError("overlay_mode must be 'boundary', 'filled', or 'none'")

    # formatting
    ax.set_xlim(x0_px, x1_px)
    ax.set_ylim(y1_px, y0_px)
    ax.set_aspect("equal")
    ax.set_title(f"{sample}: {channel}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

    # scale bar
    if show_scale_bar:
        scale_bar_px = scale_bar_um / um_per_coord

        x_range = x1_px - x0_px
        y_range = y1_px - y0_px

        pad_x = 0.05 * x_range
        pad_y = 0.05 * y_range

        if scale_bar_location == "lower right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "lower left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "upper right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        elif scale_bar_location == "upper left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        else:
            raise ValueError(
                "scale_bar_location must be 'lower right', 'lower left', "
                "'upper right', or 'upper left'"
            )

        ax.plot(
            [x_start, x_end],
            [y, y],
            color=scale_bar_color,
            linewidth=scale_bar_linewidth,
            solid_capstyle="butt"
        )

        ax.text(
            (x_start + x_end) / 2,
            text_y,
            f"{scale_bar_um} µm",
            color=scale_bar_color,
            ha="center",
            va=va,
            fontsize=scale_bar_fontsize
        )

    # legend
    if show_legend and overlay_mode != "none":
        if overlay_mode == "boundary":
            handles = [
                Line2D([0], [0], color=celltype_colors_plot[ct], lw=2, label=ct)
                for ct in present_celltypes
            ]
        else:
            handles = [
                mpatches.Patch(color=celltype_colors_plot[ct], label=ct)
                for ct in present_celltypes
            ]

        ax.legend(
            handles=handles,
            title="Cell type",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0
        )

    plt.tight_layout()

    # save
    if save_label is None:
        save_label = f"ROI_DAPI_{overlay_mode}_{nuclei_display}"

    out_file = out_dir / f"{sample}_{save_label}.pdf"

    fig.savefig(
        out_file,
        dpi=dpi,
        bbox_inches="tight",
        transparent=True,
    )

    print(f"Saved: {out_file}")

    return fig, ax, out_file


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep3_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep3"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()

# Define the values used for `highlight_celltypes`.
highlight_celltypes = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `celltype_colors_plot`.
celltype_colors_plot = {
    "Basal VCMs":               "#2C7BB6",  # blue
    "IFN-associated VCMs":      "#FFD92F",  # gold
    "Stressed VCMs":            "#D7191C",  # red
    "Remodelled VCMs":           "#1A9641",  # green
    "Myeloid":                 "#FF7A00",  # orange
    "Lymphoid":                  "#7B2CFF",  # purple
    "Other": "lightgrey"
}

# Convert values to the required data type.
cell_shapes_roi["celltype_plot"] = cell_shapes_roi["celltype4"].astype(str).where(
    cell_shapes_roi["celltype4"].astype(str).isin(highlight_celltypes),
    "Other"
)

# Compute and store `cell_shapes_roi['plot_color']`.
cell_shapes_roi["plot_color"] = cell_shapes_roi["celltype_plot"].map(celltype_colors_plot)

cell_shapes_roi.shape


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Run `plot_roi_dapi` for this analysis step.
plot_roi_dapi(
    sdata=sdata,
    cell_shapes_roi_px=cell_shapes_roi_px,
    celltype_colors_plot=celltype_colors_plot,
    sample=sample,
    scale="scale0",
    out_dir=out_dir,
    x0_px=x0_px, x1_px=x1_px,
    y0_px=y0_px, y1_px=y1_px,
    overlay_mode="filled",
    fill_alpha=0.65,
    nuclei_display="gray",
    boundary_linewidth=0.1,
    show_scale_bar=True,
    scale_bar_um=50,
    save_label="filled_DAPI_with_scalebar_darker"
)


In [ ]:
# Select the required subset and store it as `img`.
img = sdata.images["morphology_focus"]

# Repeat the following operation for each item in the selected collection.
for scale in img.keys():
    arr = img[scale]["image"]
    print(scale, arr.shape)


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep1_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep1"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()

# Define the values used for `highlight_celltypes`.
highlight_celltypes = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `celltype_colors_plot`.
celltype_colors_plot = {
    "Basal VCMs":               "#2C7BB6",  # blue
    "IFN-associated VCMs":      "#FFD92F",  # gold
    "Stressed VCMs":            "#D7191C",  # red
    "Remodelled VCMs":           "#1A9641",  # green
    "Myeloid":                 "#FF7A00",  # orange
    "Lymphoid":                  "#7B2CFF",  # purple
    "Other": "lightgrey"
}

# Convert values to the required data type.
cell_shapes_roi["celltype_plot"] = cell_shapes_roi["celltype4"].astype(str).where(
    cell_shapes_roi["celltype4"].astype(str).isin(highlight_celltypes),
    "Other"
)

# Compute and store `cell_shapes_roi['plot_color']`.
cell_shapes_roi["plot_color"] = cell_shapes_roi["celltype_plot"].map(celltype_colors_plot)

cell_shapes_roi.shape


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Run `plot_roi_dapi` for this analysis step.
plot_roi_dapi(
    sdata=sdata,
    cell_shapes_roi_px=cell_shapes_roi_px,
    celltype_colors_plot=celltype_colors_plot,
    sample=sample,
    scale="scale0",
    out_dir=out_dir,
    x0_px=x0_px, x1_px=x1_px,
    y0_px=y0_px, y1_px=y1_px,
    overlay_mode="filled",
    fill_alpha=0.65,
    nuclei_display="gray",
    boundary_linewidth=0,
    show_scale_bar=True,
    scale_bar_um=50,
    save_label="_ROI2_zoomedin_darker"
)


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep4_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep4"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()

# Define the values used for `highlight_celltypes`.
highlight_celltypes = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `celltype_colors_plot`.
celltype_colors_plot = {
    "Basal VCMs":               "#2C7BB6",  # blue
    "IFN-associated VCMs":      "#FFD92F",  # gold
    "Stressed VCMs":            "#D7191C",  # red
    "Remodelled VCMs":           "#1A9641",  # green
    "Myeloid":                 "#FF7A00",  # orange
    "Lymphoid":                  "#7B2CFF",  # purple
    "Other": "lightgrey"
}

# Convert values to the required data type.
cell_shapes_roi["celltype_plot"] = cell_shapes_roi["celltype4"].astype(str).where(
    cell_shapes_roi["celltype4"].astype(str).isin(highlight_celltypes),
    "Other"
)

# Compute and store `cell_shapes_roi['plot_color']`.
cell_shapes_roi["plot_color"] = cell_shapes_roi["celltype_plot"].map(celltype_colors_plot)

cell_shapes_roi.shape


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Run `plot_roi_dapi` for this analysis step.
plot_roi_dapi(
    sdata=sdata,
    cell_shapes_roi_px=cell_shapes_roi_px,
    celltype_colors_plot=celltype_colors_plot,
    sample=sample,
    scale="scale0",
    out_dir=out_dir,
    x0_px=x0_px, x1_px=x1_px,
    y0_px=y0_px, y1_px=y1_px,
    overlay_mode="filled",
    fill_alpha=0.65,
    nuclei_display="gray",
    boundary_linewidth=0,
    show_scale_bar=True,
    scale_bar_um=50,
    save_label="ROI3_filled_DAPI_with_scalebar_darker"
)


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()

# Define the values used for `highlight_celltypes`.
highlight_celltypes = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `celltype_colors_plot`.
celltype_colors_plot = {
    "Basal VCMs":               "#2C7BB6",  # blue
    "IFN-associated VCMs":      "#FFD92F",  # gold
    "Stressed VCMs":            "#D7191C",  # red
    "Remodelled VCMs":           "#1A9641",  # green
    "Myeloid":                 "#FF7A00",  # orange
    "Lymphoid":                  "#7B2CFF",  # purple
    "Other": "lightgrey"
}

# Convert values to the required data type.
cell_shapes_roi["celltype_plot"] = cell_shapes_roi["celltype4"].astype(str).where(
    cell_shapes_roi["celltype4"].astype(str).isin(highlight_celltypes),
    "Other"
)

# Compute and store `cell_shapes_roi['plot_color']`.
cell_shapes_roi["plot_color"] = cell_shapes_roi["celltype_plot"].map(celltype_colors_plot)

cell_shapes_roi.shape


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Run `plot_roi_dapi` for this analysis step.
plot_roi_dapi(
    sdata=sdata,
    cell_shapes_roi_px=cell_shapes_roi_px,
    celltype_colors_plot=celltype_colors_plot,
    sample=sample,
    scale="scale0",
    out_dir=out_dir,
    x0_px=x0_px, x1_px=x1_px,
    y0_px=y0_px, y1_px=y1_px,
    overlay_mode="filled",
    fill_alpha=0.65,
    nuclei_display="gray",
    boundary_linewidth=0,
    show_scale_bar=True,
    scale_bar_um=50,
    save_label="ROI1_filled_DAPI_with_scalebar_darker"
)


In [ ]:
# Define the values used for `celltype_order`.
celltype_order = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Remodelled VCMs",
    "Myeloid",
    "Lymphoid",
    "Other",
]

# Define the values used for `celltype_colors_plot`.
celltype_colors_plot = {
    "Basal VCMs":          "#2C7BB6",
    "IFN-associated VCMs": "#FFD92F",
    "Stressed VCMs":       "#D7191C",
    "Remodelled VCMs":     "#1A9641",
    "Myeloid":             "#FF7A00",
    "Lymphoid":            "#7B2CFF",
    "Other":               "lightgrey",
}

# Set `highlight_celltypes` for the following analysis.
highlight_celltypes = [
    ct for ct in celltype_order
    if ct != "Other"
]


In [ ]:
# Set `present_celltypes` for the following analysis.
present_celltypes = [
    ct for ct in celltype_order
    if ct in cell_shapes_roi_px["celltype_plot"].values
]


In [ ]:
# Make the barplot categories identical to those in the spatial plots
composition_df = adata_roi.obs[["name", "celltype4"]].copy()

# Convert values to the required data type.
composition_df["celltype_plot"] = (
    composition_df["celltype4"]
    .astype(str)
    .where(
        composition_df["celltype4"].astype(str).isin(highlight_celltypes),
        "Other"
    )
)

# Count cells
composition_counts = (
    composition_df
    .groupby(["name", "celltype_plot"], observed=True)
    .size()
    .unstack(fill_value=0)
)

# Force BOTH sample order and cell-type order
sample_order = ["BE_rep1", "BE_rep2", "BE_rep3", "BE_rep4"]

# Compute and store `composition_counts`.
composition_counts = composition_counts.reindex(
    index=sample_order,
    columns=celltype_order,
    fill_value=0
)

# Convert to percentage
composition_pct = (
    composition_counts
    .div(composition_counts.sum(axis=1), axis=0)
    * 100
)

composition_pct


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(5, 5))

# x positions for the ROIs
x = np.arange(len(composition_pct.index))

# Start stacking from the bottom
bottom = np.zeros(len(composition_pct))

# Reverse the plotting order so that the TOP of the bar
# follows the SAME order as the legend
for ct in reversed(celltype_order):
    values = composition_pct[ct].values

    ax.bar(
        x,
        values,
        bottom=bottom,
        width=0.75,
        color=celltype_colors_plot[ct],
        label=ct
    )

    bottom += values


# -------------------------
# Axis formatting
# -------------------------

# Label the y-axis.
ax.set_ylabel("Cell-type composition (%)")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 100)

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks(x)
# Run `ax.set_xticklabels` for this analysis step.
ax.set_xticklabels(
    composition_pct.index,
    rotation=45,
    ha="right"
)

# NO GRID
ax.grid(False)
# Run `ax.grid` for this analysis step.
ax.grid(False, which="both", axis="both")


# -------------------------
# Legend in desired order
# -------------------------

# Set `handles` for the following analysis.
handles = [
    mpatches.Patch(
        color=celltype_colors_plot[ct],
        label=ct
    )
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=handles,
    title="Cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Calculate `out_file` from the existing values.
out_file = out_dir / "ROI_celltype_composition.pdf"

# Save the completed figure to disk.
fig.savefig(
    out_file,
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(5, 5))

# Compute and store `x`.
x = np.arange(len(composition_pct.index))

# Compute and store `bottom`.
bottom = np.zeros(len(composition_pct))

# Plot from bottom upwards
for ct in reversed(celltype_order):
    values = composition_pct[ct].values

    ax.bar(
        x,
        values,
        bottom=bottom,
        width=0.75,
        color=celltype_colors_plot[ct]
    )

    # Percentage labels
    for i, value in enumerate(values):
        if value >= 3:
            ax.text(
                x[i],
                bottom[i] + value / 2,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=8,
                color="black"
            )

    bottom += values


# -------------------------
# Axis formatting
# -------------------------

# Label the y-axis.
ax.set_ylabel("Cell-type composition (%)")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 100)

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks(x)
# Run `ax.set_xticklabels` for this analysis step.
ax.set_xticklabels(
    composition_pct.index,
    rotation=45,
    ha="right"
)

# No grid
ax.grid(False)


# -------------------------
# Legend
# -------------------------

# Set `handles` for the following analysis.
handles = [
    mpatches.Patch(
        color=celltype_colors_plot[ct],
        label=ct
    )
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=handles,
    title="Cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Calculate `out_file` from the existing values.
out_file = out_dir / "ROI_celltype_composition_withpercentage.pdf"

# Save the completed figure to disk.
fig.savefig(
    out_file,
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Repeat the following operation for each item in the selected collection.
for sample in composition_pct.index:

    fig, ax = plt.subplots(figsize=(1.2, 8))

    bottom = 0

    # Plot in reverse so the visual top->bottom order
    # matches the legend top->bottom order
    for ct in reversed(celltype_order):

        value = composition_pct.loc[sample, ct]

        ax.bar(
            0,
            value,
            bottom=bottom,
            width=0.65,
            color=celltype_colors_plot[ct]
        )

        # Add percentage in the centre of the segment
        if value >= 3:
            ax.text(
                0,
                bottom + value / 2,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=16,
                color="black"
            )

        bottom += value

    # -------------------------
    # Formatting
    # -------------------------

    ax.set_ylim(0, 100)
    ax.set_ylabel("Cell-type composition (%)")
    ax.set_xlabel("")

    ax.set_xticks([0])
    ax.set_xticklabels([sample])

    # No grid
    ax.grid(False)

    # -------------------------
    # Legend
    # -------------------------

    handles = [
        mpatches.Patch(
            color=celltype_colors_plot[ct],
            label=ct
        )
        for ct in celltype_order
    ]

    ax.legend(
        handles=handles,
        title="Cell type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()

    # -------------------------
    # Save each sample separately
    # -------------------------

    out_file = out_dir / f"{sample}_celltype_composition.pdf"

    fig.savefig(
        out_file,
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.show()
    plt.close(fig)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Repeat the following operation for each item in the selected collection.
for sample in composition_pct.index:

    # Get values in the desired cell-type order
    values = composition_pct.loc[sample, celltype_order]

    # Optional: remove cell types with 0%
    mask = values > 0
    values_plot = values[mask]
    celltypes_plot = values.index[mask]

    colors_plot = [
        celltype_colors_plot[ct]
        for ct in celltypes_plot
    ]

    # -------------------------
    # Pie chart
    # -------------------------

    fig, ax = plt.subplots(figsize=(8, 8))

    wedges, texts, autotexts = ax.pie(
        values_plot,
        colors=colors_plot,
        startangle=90,
        counterclock=False,

        # Only show percentage for slices >= 3%
        autopct=lambda pct: f"{pct:.1f}%" if pct >= 3 else "",

        textprops={
            "fontsize": 14,
            "color": "black"
        },

        wedgeprops={
            "edgecolor": "white",
            "linewidth": 1
        }
    )

    # Percentage font size
    for autotext in autotexts:
        autotext.set_fontsize(14)

    # Make pie circular
    ax.axis("equal")

    ax.set_title(
        f"{sample}",
        fontsize=16
    )

    # -------------------------
    # Legend
    # -------------------------

    handles = [
        mpatches.Patch(
            color=celltype_colors_plot[ct],
            label=ct
        )
        for ct in celltype_order
        if composition_pct.loc[sample, ct] > 0
    ]

    ax.legend(
        handles=handles,
        title="Cell type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()

    # -------------------------
    # Save
    # -------------------------

    out_file = out_dir / f"{sample}_celltype_composition_pie.pdf"

    fig.savefig(
        out_file,
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.show()
    plt.close(fig)


## WT/PBS morphology zooms and composition




In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Set `sc.settings.figdir` for the following analysis.
sc.settings.figdir = "Figures_forpaper/new_colours/noWT3spatial/ROIonscatter"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Compute and store `out_dir`.
out_dir = Path("Figures_forpaper/new_colours/noWT3spatial/ROIonscatter")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad")


In [ ]:
# Define the values used for `roi_centers`.
roi_centers = {
    #"PBS_rep1": (1681, 3271),
    "PBS_rep2": (1404, 4964, 100000),
    #"PBS_rep2": (2950, 3964, 150000),
    #"PBS_rep3": (3261, 3327),
    #"PBS_rep4": (1336, 1119),
    #"WT_rep1": (3913, 908, 150000),
    "WT_rep2": (1425, 3652, 100000),   
    #"WT_rep4": (3636,3576, 150000),
    #"WT_rep5": (2090, 4000)
}


In [ ]:
# Define the values used for `roi_squares`.
roi_squares = {}

# Repeat the following operation for each item in the selected collection.
for sample, (x_center, y_center, roi_area_um2) in roi_centers.items():

    roi_side_um = np.sqrt(roi_area_um2)
    half = roi_side_um / 2

    roi_squares[sample] = {
        "x_center": x_center,
        "y_center": y_center,
        "x_min": x_center - half,
        "x_max": x_center + half,
        "y_min": y_center - half,
        "y_max": y_center + half,
        "side_um": roi_side_um,
        "area_um2": roi_area_um2,
    }

# Create a DataFrame for downstream analysis.
roi_squares_df = pd.DataFrame.from_dict(roi_squares, orient="index")
roi_squares_df


In [ ]:
# Select the required subset and store it as `adata.obs['x']`.
adata.obs["x"] = adata.obsm["spatial"][:, 0]
# Select the required subset and store it as `adata.obs['y']`.
adata.obs["y"] = adata.obsm["spatial"][:, 1]


In [ ]:
# Set `adata.obs['in_roi']` for the following analysis.
adata.obs["in_roi"] = False
# Set `adata.obs['roi_id']` for the following analysis.
adata.obs["roi_id"] = np.nan
# Set `adata.obs['roi_area_um2']` for the following analysis.
adata.obs["roi_area_um2"] = np.nan

# Repeat the following operation for each item in the selected collection.
for sample, roi in roi_squares_df.iterrows():

    mask = (
        (adata.obs["name"] == sample) &
        (adata.obs["x"] >= roi["x_min"]) &
        (adata.obs["x"] <= roi["x_max"]) &
        (adata.obs["y"] >= roi["y_min"]) &
        (adata.obs["y"] <= roi["y_max"])
    )

    adata.obs.loc[mask, "in_roi"] = True
    adata.obs.loc[mask, "roi_id"] = sample
    adata.obs.loc[mask, "roi_area_um2"] = roi["area_um2"]


In [ ]:
# Make an independent copy of the selected data.
adata_roi = adata[adata.obs["in_roi"]].copy()


In [ ]:
# Inspect the number of observations in each category.
adata_roi.obs['name'].value_counts()


In [ ]:
# Reset the DataFrame index after reshaping or filtering.
celltype_counts_sample = (
    adata_roi.obs
    .groupby(["name", "celltype4"])
    .size()
    .reset_index(name="n_cells")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head(20)


In [ ]:
# Calculate `celltype_counts_sample['fraction']` from the existing values.
celltype_counts_sample["fraction"] = (
    celltype_counts_sample["n_cells"] /
    celltype_counts_sample.groupby("name")["n_cells"].transform("sum")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define `plot_roi_dapi()` for reuse in the analysis below.
def plot_roi_dapi(
    sdata,
    cell_shapes_roi_px,
    celltype_colors_plot,
    sample,
    out_dir,
    x0_px, x1_px, y0_px, y1_px,
    scale="scale2",
    channel="DAPI",
    overlay_mode="boundary",   # "boundary", "filled", or "none"
    nuclei_display="gray",     # "gray" or "blue"
    fill_alpha=0.5,
    boundary_linewidth=0.4,
    filled_edgecolor="white",
    filled_linewidth=0.15,
    show_legend=True,
    figsize=(6, 6),
    dpi=300,

    # scale bar options
    show_scale_bar=True,
    scale_bar_um=50,
    um_per_coord=0.2125,
    scale_bar_color="white",
    scale_bar_linewidth=3,
    scale_bar_fontsize=8,
    scale_bar_location="lower right",

    # saving option
    save_label=None
):
    """
    Plot a single ROI for the DAPI channel with flexible cell overlay options.

    Important:
    This assumes x0_px/x1_px/y0_px/y1_px and cell_shapes_roi_px are in
    scale0 pixel coordinates, matching the x/y coordinates of the SpatialData image.
    """

    # get image object
    img_xr = sdata.images["morphology_focus"][scale]["image"]

    # copy and assign colors
    cell_shapes_roi_px = cell_shapes_roi_px.copy()
    cell_shapes_roi_px["plot_color"] = cell_shapes_roi_px["celltype_plot"].map(celltype_colors_plot)

    # crop DAPI image
    img_crop = img_xr.sel(
        c=channel,
        x=slice(x0_px, x1_px),
        y=slice(y0_px, y1_px)
    ).data.compute()

    # contrast scaling
    vmin, vmax = np.percentile(img_crop, [2, 99.5])
    img_norm = np.clip((img_crop - vmin) / (vmax - vmin + 1e-8), 0, 1)

    fig, ax = plt.subplots(figsize=figsize)

    # plot nuclei channel
    if nuclei_display == "gray":
        ax.imshow(
            img_crop,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    elif nuclei_display == "blue":
        rgb = np.zeros((*img_norm.shape, 3), dtype=float)
        rgb[..., 2] = img_norm
        ax.imshow(
            rgb,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    else:
        raise ValueError("nuclei_display must be 'gray' or 'blue'")

    # overlay polygons
    present_celltypes = [
        ct for ct in celltype_colors_plot
        if ct in cell_shapes_roi_px["celltype_plot"].values
    ]

    if overlay_mode == "boundary":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.boundary.plot(
                    ax=ax,
                    edgecolor=celltype_colors_plot[ct],
                    linewidth=boundary_linewidth
                )

    elif overlay_mode == "filled":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.plot(
                    ax=ax,
                    color=celltype_colors_plot[ct],
                    edgecolor=filled_edgecolor,
                    linewidth=filled_linewidth,
                    alpha=fill_alpha
                )

    elif overlay_mode == "none":
        pass

    else:
        raise ValueError("overlay_mode must be 'boundary', 'filled', or 'none'")

    # formatting
    ax.set_xlim(x0_px, x1_px)
    ax.set_ylim(y1_px, y0_px)
    ax.set_aspect("equal")
    ax.set_title(f"{sample}: {channel}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

    # scale bar
    if show_scale_bar:
        scale_bar_px = scale_bar_um / um_per_coord

        x_range = x1_px - x0_px
        y_range = y1_px - y0_px

        pad_x = 0.05 * x_range
        pad_y = 0.05 * y_range

        if scale_bar_location == "lower right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "lower left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "upper right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        elif scale_bar_location == "upper left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        else:
            raise ValueError(
                "scale_bar_location must be 'lower right', 'lower left', "
                "'upper right', or 'upper left'"
            )

        ax.plot(
            [x_start, x_end],
            [y, y],
            color=scale_bar_color,
            linewidth=scale_bar_linewidth,
            solid_capstyle="butt"
        )

        ax.text(
            (x_start + x_end) / 2,
            text_y,
            f"{scale_bar_um} µm",
            color=scale_bar_color,
            ha="center",
            va=va,
            fontsize=scale_bar_fontsize
        )

    # legend
    if show_legend and overlay_mode != "none":
        if overlay_mode == "boundary":
            handles = [
                Line2D([0], [0], color=celltype_colors_plot[ct], lw=2, label=ct)
                for ct in present_celltypes
            ]
        else:
            handles = [
                mpatches.Patch(color=celltype_colors_plot[ct], label=ct)
                for ct in present_celltypes
            ]

        ax.legend(
            handles=handles,
            title="Cell type",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0
        )

    plt.tight_layout()

    # save
    if save_label is None:
        save_label = f"ROI_DAPI_{overlay_mode}_{nuclei_display}"

    out_file = out_dir / f"{sample}_{save_label}.pdf"

    fig.savefig(
        out_file,
        dpi=dpi,
        bbox_inches="tight",
        transparent=True,
    )

    print(f"Saved: {out_file}")

    return fig, ax, out_file


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/PBS_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "PBS_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
PBS_rep2 = adata[adata.obs['name']=="PBS_rep2"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "PBS_rep2"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
PBS_rep2_plot = PBS_rep2.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = PBS_rep2_plot.obs["celltype4"].astype(str)

# Set `PBS_rep2_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
PBS_rep2_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        PBS_rep2_plot.obs["celltype4"].isna(),
        PBS_rep2_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
PBS_rep2_plot = PBS_rep2_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    PBS_rep2_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("PBS_rep2: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "PBS_rep2_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/WT_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "WT_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
WT_rep2 = adata[adata.obs['name']=="WT_rep2"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "WT_rep2"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
WT_rep2_plot = WT_rep2.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = WT_rep2_plot.obs["celltype4"].astype(str)

# Set `WT_rep2_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
WT_rep2_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        WT_rep2_plot.obs["celltype4"].isna(),
        WT_rep2_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
WT_rep2_plot = WT_rep2_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    WT_rep2_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("WT_rep2: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "WT_rep2_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Define the values used for `celltype_order`.
celltype_order = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Remodelled VCMs",
    "Myeloid",
    "Lymphoid",
    "Other",
]

# Define the values used for `celltype_colors_plot`.
celltype_colors_plot = {
    "Basal VCMs":          "#2C7BB6",
    "IFN-associated VCMs": "#FFD92F",
    "Stressed VCMs":       "#D7191C",
    "Remodelled VCMs":     "#1A9641",
    "Myeloid":             "#FF7A00",
    "Lymphoid":            "#7B2CFF",
    "Other":               "lightgrey",
}

# Set `highlight_celltypes` for the following analysis.
highlight_celltypes = [
    ct for ct in celltype_order
    if ct != "Other"
]


In [ ]:
# Make the barplot categories identical to those in the spatial plots
composition_df = adata_roi.obs[["name", "celltype4"]].copy()

# Convert values to the required data type.
composition_df["celltype_plot"] = (
    composition_df["celltype4"]
    .astype(str)
    .where(
        composition_df["celltype4"].astype(str).isin(highlight_celltypes),
        "Other"
    )
)

# Count cells
composition_counts = (
    composition_df
    .groupby(["name", "celltype_plot"], observed=True)
    .size()
    .unstack(fill_value=0)
)

# Force BOTH sample order and cell-type order
sample_order = ["WT_rep2", "PBS_rep2"]

# Compute and store `composition_counts`.
composition_counts = composition_counts.reindex(
    index=sample_order,
    columns=celltype_order,
    fill_value=0
)

# Convert to percentage
composition_pct = (
    composition_counts
    .div(composition_counts.sum(axis=1), axis=0)
    * 100
)

composition_pct


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(5, 7))

# x positions for the ROIs
x = np.arange(len(composition_pct.index))

# Start stacking from the bottom
bottom = np.zeros(len(composition_pct))

# Reverse the plotting order so that the TOP of the bar
# follows the SAME order as the legend
for ct in reversed(celltype_order):
    values = composition_pct[ct].values

    ax.bar(
        x,
        values,
        bottom=bottom,
        width=0.75,
        color=celltype_colors_plot[ct],
        label=ct
    )

    bottom += values


# -------------------------
# Axis formatting
# -------------------------

# Label the y-axis.
ax.set_ylabel("Cell-type composition (%)")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 100)

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks(x)
# Run `ax.set_xticklabels` for this analysis step.
ax.set_xticklabels(
    composition_pct.index,
    rotation=45,
    ha="right"
)

# NO GRID
ax.grid(False)
# Run `ax.grid` for this analysis step.
ax.grid(False, which="both", axis="both")


# -------------------------
# Legend in desired order
# -------------------------

# Set `handles` for the following analysis.
handles = [
    mpatches.Patch(
        color=celltype_colors_plot[ct],
        label=ct
    )
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=handles,
    title="Cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Calculate `out_file` from the existing values.
out_file = out_dir / "PBS_WT_ROI_celltype_composition.pdf"

# Save the completed figure to disk.
fig.savefig(
    out_file,
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(5, 7))

# Compute and store `x`.
x = np.arange(len(composition_pct.index))

# Compute and store `bottom`.
bottom = np.zeros(len(composition_pct))

# Plot from bottom upwards
for ct in reversed(celltype_order):
    values = composition_pct[ct].values

    ax.bar(
        x,
        values,
        bottom=bottom,
        width=0.75,
        color=celltype_colors_plot[ct]
    )

    # Percentage labels
    for i, value in enumerate(values):
        if value >= 3:
            ax.text(
                x[i],
                bottom[i] + value / 2,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=8,
                color="black"
            )

    bottom += values


# -------------------------
# Axis formatting
# -------------------------

# Label the y-axis.
ax.set_ylabel("Cell-type composition (%)")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 100)

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks(x)
# Run `ax.set_xticklabels` for this analysis step.
ax.set_xticklabels(
    composition_pct.index,
    rotation=45,
    ha="right"
)

# No grid
ax.grid(False)


# -------------------------
# Legend
# -------------------------

# Set `handles` for the following analysis.
handles = [
    mpatches.Patch(
        color=celltype_colors_plot[ct],
        label=ct
    )
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=handles,
    title="Cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Calculate `out_file` from the existing values.
out_file = out_dir / "PBS_WT_ROI_celltype_composition_withpercentage.pdf"

# Save the completed figure to disk.
fig.savefig(
    out_file,
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Repeat the following operation for each item in the selected collection.
for sample in composition_pct.index:

    fig, ax = plt.subplots(figsize=(1.2, 8))

    bottom = 0

    # Plot in reverse so the visual top->bottom order
    # matches the legend top->bottom order
    for ct in reversed(celltype_order):

        value = composition_pct.loc[sample, ct]

        ax.bar(
            0,
            value,
            bottom=bottom,
            width=0.65,
            color=celltype_colors_plot[ct]
        )

        # Add percentage in the centre of the segment
        if value >= 3:
            ax.text(
                0,
                bottom + value / 2,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=16,
                color="black"
            )

        bottom += value

    # -------------------------
    # Formatting
    # -------------------------

    ax.set_ylim(0, 100)
    ax.set_ylabel("Cell-type composition (%)")
    ax.set_xlabel("")

    ax.set_xticks([0])
    ax.set_xticklabels([sample])

    # No grid
    ax.grid(False)

    # -------------------------
    # Legend
    # -------------------------

    handles = [
        mpatches.Patch(
            color=celltype_colors_plot[ct],
            label=ct
        )
        for ct in celltype_order
    ]

    ax.legend(
        handles=handles,
        title="Cell type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()

    # -------------------------
    # Save each sample separately
    # -------------------------

    out_file = out_dir / f"{sample}_celltype_composition.pdf"

    fig.savefig(
        out_file,
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.show()
    plt.close(fig)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Repeat the following operation for each item in the selected collection.
for sample in composition_pct.index:

    # Get values in the desired cell-type order
    values = composition_pct.loc[sample, celltype_order]

    # Optional: remove cell types with 0%
    mask = values > 0
    values_plot = values[mask]
    celltypes_plot = values.index[mask]

    colors_plot = [
        celltype_colors_plot[ct]
        for ct in celltypes_plot
    ]

    # -------------------------
    # Pie chart
    # -------------------------

    fig, ax = plt.subplots(figsize=(8, 8))

    wedges, texts, autotexts = ax.pie(
        values_plot,
        colors=colors_plot,
        startangle=90,
        counterclock=False,

        # Only show percentage for slices >= 3%
        autopct=lambda pct: f"{pct:.1f}%" if pct >= 3 else "",

        textprops={
            "fontsize": 14,
            "color": "black"
        },

        wedgeprops={
            "edgecolor": "white",
            "linewidth": 1
        }
    )

    # Percentage font size
    for autotext in autotexts:
        autotext.set_fontsize(14)

    # Make pie circular
    ax.axis("equal")

    ax.set_title(
        f"{sample}",
        fontsize=16
    )

    # -------------------------
    # Legend
    # -------------------------

    handles = [
        mpatches.Patch(
            color=celltype_colors_plot[ct],
            label=ct
        )
        for ct in celltype_order
        if composition_pct.loc[sample, ct] > 0
    ]

    ax.legend(
        handles=handles,
        title="Cell type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()

    # -------------------------
    # Save
    # -------------------------

    out_file = out_dir / f"{sample}_celltype_composition_pie.pdf"

    fig.savefig(
        out_file,
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.show()
    plt.close(fig)


## BE ROI-on-scatter — same-sized ROIs

This section preserves the same-sized BE ROI-on-scatter workflow for the selected BE replicates.
The common coordinate conversion and overlay procedure is documented once for the section rather
than repeatedly between equivalent sample blocks.


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Set `sc.settings.figdir` for the following analysis.
sc.settings.figdir = "Figures_forpaper/new_colours/noWT3spatial/ROIonscatter/samesized"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Compute and store `out_dir`.
out_dir = Path("Figures_forpaper/new_colours/noWT3spatial/ROIonscatter/samesized")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad")


In [ ]:
# Define the values used for `roi_centers`.
roi_centers = {
    "BE_rep1": (1032, 3520, 100000), #little lymphoid cluster in the middle but not perfect segmentation #ROI2 #size 50 000
    "BE_rep2":  (3070, 2347, 100000), #ROI1, 300 000
    "BE_rep3": (3694, 1151, 100000), #The good one #size 150 000
    "BE_rep4":  (1814, 2032, 100000), #ROI3 #100 000
}


In [ ]:
# Define the values used for `roi_squares`.
roi_squares = {}

# Repeat the following operation for each item in the selected collection.
for sample, (x_center, y_center, roi_area_um2) in roi_centers.items():

    roi_side_um = np.sqrt(roi_area_um2)
    half = roi_side_um / 2

    roi_squares[sample] = {
        "x_center": x_center,
        "y_center": y_center,
        "x_min": x_center - half,
        "x_max": x_center + half,
        "y_min": y_center - half,
        "y_max": y_center + half,
        "side_um": roi_side_um,
        "area_um2": roi_area_um2,
    }

# Create a DataFrame for downstream analysis.
roi_squares_df = pd.DataFrame.from_dict(roi_squares, orient="index")
roi_squares_df


In [ ]:
# Select the required subset and store it as `adata.obs['x']`.
adata.obs["x"] = adata.obsm["spatial"][:, 0]
# Select the required subset and store it as `adata.obs['y']`.
adata.obs["y"] = adata.obsm["spatial"][:, 1]


In [ ]:
# Set `adata.obs['in_roi']` for the following analysis.
adata.obs["in_roi"] = False
# Set `adata.obs['roi_id']` for the following analysis.
adata.obs["roi_id"] = np.nan
# Set `adata.obs['roi_area_um2']` for the following analysis.
adata.obs["roi_area_um2"] = np.nan

# Repeat the following operation for each item in the selected collection.
for sample, roi in roi_squares_df.iterrows():

    mask = (
        (adata.obs["name"] == sample) &
        (adata.obs["x"] >= roi["x_min"]) &
        (adata.obs["x"] <= roi["x_max"]) &
        (adata.obs["y"] >= roi["y_min"]) &
        (adata.obs["y"] <= roi["y_max"])
    )

    adata.obs.loc[mask, "in_roi"] = True
    adata.obs.loc[mask, "roi_id"] = sample
    adata.obs.loc[mask, "roi_area_um2"] = roi["area_um2"]


In [ ]:
# Make an independent copy of the selected data.
adata_roi = adata[adata.obs["in_roi"]].copy()


In [ ]:
# Inspect the number of observations in each category.
adata_roi.obs['name'].value_counts()


In [ ]:
# Reset the DataFrame index after reshaping or filtering.
celltype_counts_sample = (
    adata_roi.obs
    .groupby(["name", "celltype4"])
    .size()
    .reset_index(name="n_cells")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head(20)


In [ ]:
# Calculate `celltype_counts_sample['fraction']` from the existing values.
celltype_counts_sample["fraction"] = (
    celltype_counts_sample["n_cells"] /
    celltype_counts_sample.groupby("name")["n_cells"].transform("sum")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define `plot_roi_dapi()` for reuse in the analysis below.
def plot_roi_dapi(
    sdata,
    cell_shapes_roi_px,
    celltype_colors_plot,
    sample,
    out_dir,
    x0_px, x1_px, y0_px, y1_px,
    scale="scale2",
    channel="DAPI",
    overlay_mode="boundary",   # "boundary", "filled", or "none"
    nuclei_display="gray",     # "gray" or "blue"
    fill_alpha=0.5,
    boundary_linewidth=0.4,
    filled_edgecolor="white",
    filled_linewidth=0.15,
    show_legend=True,
    figsize=(6, 6),
    dpi=300,

    # scale bar options
    show_scale_bar=True,
    scale_bar_um=50,
    um_per_coord=0.2125,
    scale_bar_color="white",
    scale_bar_linewidth=3,
    scale_bar_fontsize=8,
    scale_bar_location="lower right",

    # saving option
    save_label=None
):
    """
    Plot a single ROI for the DAPI channel with flexible cell overlay options.

    Important:
    This assumes x0_px/x1_px/y0_px/y1_px and cell_shapes_roi_px are in
    scale0 pixel coordinates, matching the x/y coordinates of the SpatialData image.
    """

    # get image object
    img_xr = sdata.images["morphology_focus"][scale]["image"]

    # copy and assign colors
    cell_shapes_roi_px = cell_shapes_roi_px.copy()
    cell_shapes_roi_px["plot_color"] = cell_shapes_roi_px["celltype_plot"].map(celltype_colors_plot)

    # crop DAPI image
    img_crop = img_xr.sel(
        c=channel,
        x=slice(x0_px, x1_px),
        y=slice(y0_px, y1_px)
    ).data.compute()

    # contrast scaling
    vmin, vmax = np.percentile(img_crop, [2, 99.5])
    img_norm = np.clip((img_crop - vmin) / (vmax - vmin + 1e-8), 0, 1)

    fig, ax = plt.subplots(figsize=figsize)

    # plot nuclei channel
    if nuclei_display == "gray":
        ax.imshow(
            img_crop,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    elif nuclei_display == "blue":
        rgb = np.zeros((*img_norm.shape, 3), dtype=float)
        rgb[..., 2] = img_norm
        ax.imshow(
            rgb,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    else:
        raise ValueError("nuclei_display must be 'gray' or 'blue'")

    # overlay polygons
    present_celltypes = [
        ct for ct in celltype_colors_plot
        if ct in cell_shapes_roi_px["celltype_plot"].values
    ]

    if overlay_mode == "boundary":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.boundary.plot(
                    ax=ax,
                    edgecolor=celltype_colors_plot[ct],
                    linewidth=boundary_linewidth
                )

    elif overlay_mode == "filled":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.plot(
                    ax=ax,
                    color=celltype_colors_plot[ct],
                    edgecolor=filled_edgecolor,
                    linewidth=filled_linewidth,
                    alpha=fill_alpha
                )

    elif overlay_mode == "none":
        pass

    else:
        raise ValueError("overlay_mode must be 'boundary', 'filled', or 'none'")

    # formatting
    ax.set_xlim(x0_px, x1_px)
    ax.set_ylim(y1_px, y0_px)
    ax.set_aspect("equal")
    ax.set_title(f"{sample}: {channel}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

    # scale bar
    if show_scale_bar:
        scale_bar_px = scale_bar_um / um_per_coord

        x_range = x1_px - x0_px
        y_range = y1_px - y0_px

        pad_x = 0.05 * x_range
        pad_y = 0.05 * y_range

        if scale_bar_location == "lower right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "lower left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "upper right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        elif scale_bar_location == "upper left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        else:
            raise ValueError(
                "scale_bar_location must be 'lower right', 'lower left', "
                "'upper right', or 'upper left'"
            )

        ax.plot(
            [x_start, x_end],
            [y, y],
            color=scale_bar_color,
            linewidth=scale_bar_linewidth,
            solid_capstyle="butt"
        )

        ax.text(
            (x_start + x_end) / 2,
            text_y,
            f"{scale_bar_um} µm",
            color=scale_bar_color,
            ha="center",
            va=va,
            fontsize=scale_bar_fontsize
        )

    # legend
    if show_legend and overlay_mode != "none":
        if overlay_mode == "boundary":
            handles = [
                Line2D([0], [0], color=celltype_colors_plot[ct], lw=2, label=ct)
                for ct in present_celltypes
            ]
        else:
            handles = [
                mpatches.Patch(color=celltype_colors_plot[ct], label=ct)
                for ct in present_celltypes
            ]

        ax.legend(
            handles=handles,
            title="Cell type",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0
        )

    plt.tight_layout()

    # save
    if save_label is None:
        save_label = f"ROI_DAPI_{overlay_mode}_{nuclei_display}"

    out_file = out_dir / f"{sample}_{save_label}.pdf"

    fig.savefig(
        out_file,
        dpi=dpi,
        bbox_inches="tight",
        transparent=True,
    )

    print(f"Saved: {out_file}")

    return fig, ax, out_file


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep3_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep3"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
BE_rep3 = adata[adata.obs['name']=="BE_rep3"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "BE_rep3"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
BE_rep3_plot = BE_rep3.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = BE_rep3_plot.obs["celltype4"].astype(str)

# Set `BE_rep3_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
BE_rep3_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        BE_rep3_plot.obs["celltype4"].isna(),
        BE_rep3_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
BE_rep3_plot = BE_rep3_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep3_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("BE_rep3: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep3_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep1_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep1"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
BE_rep1 = adata[adata.obs['name']=="BE_rep1"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "BE_rep1"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
BE_rep1_plot = BE_rep1.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = BE_rep1_plot.obs["celltype4"].astype(str)

# Set `BE_rep1_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
BE_rep1_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        BE_rep1_plot.obs["celltype4"].isna(),
        BE_rep1_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
BE_rep1_plot = BE_rep1_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep1_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("BE_rep1: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep1_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
BE_rep2 = adata[adata.obs['name']=="BE_rep2"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "BE_rep2"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
BE_rep2_plot = BE_rep2.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = BE_rep2_plot.obs["celltype4"].astype(str)

# Set `BE_rep2_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
BE_rep2_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        BE_rep2_plot.obs["celltype4"].isna(),
        BE_rep2_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
BE_rep2_plot = BE_rep2_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep2_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("BE_rep2: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep2_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep4_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep4"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
BE_rep4 = adata[adata.obs['name']=="BE_rep4"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "BE_rep4"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
BE_rep4_plot = BE_rep4.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = BE_rep4_plot.obs["celltype4"].astype(str)

# Set `BE_rep4_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
BE_rep4_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        BE_rep4_plot.obs["celltype4"].isna(),
        BE_rep4_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
BE_rep4_plot = BE_rep4_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep4_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("BE_rep4: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep4_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


## WT/PBS ROI-on-scatter — original ROI sizes

This section preserves the WT/PBS ROI-on-scatter variants using the ROI sizes/coordinates from the
original plotting notebook. These are kept separately from the same-sized versions because they
represent distinct figure-generation choices.


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Set `sc.settings.figdir` for the following analysis.
sc.settings.figdir = "Figures_forpaper/new_colours/noWT3spatial/ROIonscatter"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Compute and store `out_dir`.
out_dir = Path("Figures_forpaper/new_colours/noWT3spatial/ROIonscatter")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad")


In [ ]:
# Define the values used for `roi_centers`.
roi_centers = {
    #"PBS_rep1": (1681, 3271),
    "PBS_rep2": (1404, 4964, 150000),
    #"PBS_rep2": (2950, 3964, 150000),
    #"PBS_rep3": (3261, 3327),
    #"PBS_rep4": (1336, 1119),
    #"WT_rep1": (3913, 908, 150000),
    "WT_rep2": (1425, 3652, 150000),   
    #"WT_rep4": (3636,3576, 150000),
    #"WT_rep5": (2090, 4000)
}


In [ ]:
# Define the values used for `roi_squares`.
roi_squares = {}

# Repeat the following operation for each item in the selected collection.
for sample, (x_center, y_center, roi_area_um2) in roi_centers.items():

    roi_side_um = np.sqrt(roi_area_um2)
    half = roi_side_um / 2

    roi_squares[sample] = {
        "x_center": x_center,
        "y_center": y_center,
        "x_min": x_center - half,
        "x_max": x_center + half,
        "y_min": y_center - half,
        "y_max": y_center + half,
        "side_um": roi_side_um,
        "area_um2": roi_area_um2,
    }

# Create a DataFrame for downstream analysis.
roi_squares_df = pd.DataFrame.from_dict(roi_squares, orient="index")
roi_squares_df


In [ ]:
# Select the required subset and store it as `adata.obs['x']`.
adata.obs["x"] = adata.obsm["spatial"][:, 0]
# Select the required subset and store it as `adata.obs['y']`.
adata.obs["y"] = adata.obsm["spatial"][:, 1]


In [ ]:
# Set `adata.obs['in_roi']` for the following analysis.
adata.obs["in_roi"] = False
# Set `adata.obs['roi_id']` for the following analysis.
adata.obs["roi_id"] = np.nan
# Set `adata.obs['roi_area_um2']` for the following analysis.
adata.obs["roi_area_um2"] = np.nan

# Repeat the following operation for each item in the selected collection.
for sample, roi in roi_squares_df.iterrows():

    mask = (
        (adata.obs["name"] == sample) &
        (adata.obs["x"] >= roi["x_min"]) &
        (adata.obs["x"] <= roi["x_max"]) &
        (adata.obs["y"] >= roi["y_min"]) &
        (adata.obs["y"] <= roi["y_max"])
    )

    adata.obs.loc[mask, "in_roi"] = True
    adata.obs.loc[mask, "roi_id"] = sample
    adata.obs.loc[mask, "roi_area_um2"] = roi["area_um2"]


In [ ]:
# Make an independent copy of the selected data.
adata_roi = adata[adata.obs["in_roi"]].copy()


In [ ]:
# Inspect the number of observations in each category.
adata_roi.obs['name'].value_counts()


In [ ]:
# Reset the DataFrame index after reshaping or filtering.
celltype_counts_sample = (
    adata_roi.obs
    .groupby(["name", "celltype4"])
    .size()
    .reset_index(name="n_cells")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head(20)


In [ ]:
# Calculate `celltype_counts_sample['fraction']` from the existing values.
celltype_counts_sample["fraction"] = (
    celltype_counts_sample["n_cells"] /
    celltype_counts_sample.groupby("name")["n_cells"].transform("sum")
)

# Inspect the first rows of the resulting table.
celltype_counts_sample.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define `plot_roi_dapi()` for reuse in the analysis below.
def plot_roi_dapi(
    sdata,
    cell_shapes_roi_px,
    celltype_colors_plot,
    sample,
    out_dir,
    x0_px, x1_px, y0_px, y1_px,
    scale="scale2",
    channel="DAPI",
    overlay_mode="boundary",   # "boundary", "filled", or "none"
    nuclei_display="gray",     # "gray" or "blue"
    fill_alpha=0.5,
    boundary_linewidth=0.4,
    filled_edgecolor="white",
    filled_linewidth=0.15,
    show_legend=True,
    figsize=(6, 6),
    dpi=300,

    # scale bar options
    show_scale_bar=True,
    scale_bar_um=50,
    um_per_coord=0.2125,
    scale_bar_color="white",
    scale_bar_linewidth=3,
    scale_bar_fontsize=8,
    scale_bar_location="lower right",

    # saving option
    save_label=None
):
    """
    Plot a single ROI for the DAPI channel with flexible cell overlay options.

    Important:
    This assumes x0_px/x1_px/y0_px/y1_px and cell_shapes_roi_px are in
    scale0 pixel coordinates, matching the x/y coordinates of the SpatialData image.
    """

    # get image object
    img_xr = sdata.images["morphology_focus"][scale]["image"]

    # copy and assign colors
    cell_shapes_roi_px = cell_shapes_roi_px.copy()
    cell_shapes_roi_px["plot_color"] = cell_shapes_roi_px["celltype_plot"].map(celltype_colors_plot)

    # crop DAPI image
    img_crop = img_xr.sel(
        c=channel,
        x=slice(x0_px, x1_px),
        y=slice(y0_px, y1_px)
    ).data.compute()

    # contrast scaling
    vmin, vmax = np.percentile(img_crop, [2, 99.5])
    img_norm = np.clip((img_crop - vmin) / (vmax - vmin + 1e-8), 0, 1)

    fig, ax = plt.subplots(figsize=figsize)

    # plot nuclei channel
    if nuclei_display == "gray":
        ax.imshow(
            img_crop,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    elif nuclei_display == "blue":
        rgb = np.zeros((*img_norm.shape, 3), dtype=float)
        rgb[..., 2] = img_norm
        ax.imshow(
            rgb,
            extent=[x0_px, x1_px, y1_px, y0_px]
        )

    else:
        raise ValueError("nuclei_display must be 'gray' or 'blue'")

    # overlay polygons
    present_celltypes = [
        ct for ct in celltype_colors_plot
        if ct in cell_shapes_roi_px["celltype_plot"].values
    ]

    if overlay_mode == "boundary":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.boundary.plot(
                    ax=ax,
                    edgecolor=celltype_colors_plot[ct],
                    linewidth=boundary_linewidth
                )

    elif overlay_mode == "filled":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.plot(
                    ax=ax,
                    color=celltype_colors_plot[ct],
                    edgecolor=filled_edgecolor,
                    linewidth=filled_linewidth,
                    alpha=fill_alpha
                )

    elif overlay_mode == "none":
        pass

    else:
        raise ValueError("overlay_mode must be 'boundary', 'filled', or 'none'")

    # formatting
    ax.set_xlim(x0_px, x1_px)
    ax.set_ylim(y1_px, y0_px)
    ax.set_aspect("equal")
    ax.set_title(f"{sample}: {channel}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

    # scale bar
    if show_scale_bar:
        scale_bar_px = scale_bar_um / um_per_coord

        x_range = x1_px - x0_px
        y_range = y1_px - y0_px

        pad_x = 0.05 * x_range
        pad_y = 0.05 * y_range

        if scale_bar_location == "lower right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "lower left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y1_px - pad_y
            text_y = y - 0.025 * y_range
            va = "bottom"

        elif scale_bar_location == "upper right":
            x_end = x1_px - pad_x
            x_start = x_end - scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        elif scale_bar_location == "upper left":
            x_start = x0_px + pad_x
            x_end = x_start + scale_bar_px
            y = y0_px + pad_y
            text_y = y + 0.025 * y_range
            va = "top"

        else:
            raise ValueError(
                "scale_bar_location must be 'lower right', 'lower left', "
                "'upper right', or 'upper left'"
            )

        ax.plot(
            [x_start, x_end],
            [y, y],
            color=scale_bar_color,
            linewidth=scale_bar_linewidth,
            solid_capstyle="butt"
        )

        ax.text(
            (x_start + x_end) / 2,
            text_y,
            f"{scale_bar_um} µm",
            color=scale_bar_color,
            ha="center",
            va=va,
            fontsize=scale_bar_fontsize
        )

    # legend
    if show_legend and overlay_mode != "none":
        if overlay_mode == "boundary":
            handles = [
                Line2D([0], [0], color=celltype_colors_plot[ct], lw=2, label=ct)
                for ct in present_celltypes
            ]
        else:
            handles = [
                mpatches.Patch(color=celltype_colors_plot[ct], label=ct)
                for ct in present_celltypes
            ]

        ax.legend(
            handles=handles,
            title="Cell type",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0
        )

    plt.tight_layout()

    # save
    if save_label is None:
        save_label = f"ROI_DAPI_{overlay_mode}_{nuclei_display}"

    out_file = out_dir / f"{sample}_{save_label}.pdf"

    fig.savefig(
        out_file,
        dpi=dpi,
        bbox_inches="tight",
        transparent=True,
    )

    print(f"Saved: {out_file}")

    return fig, ax, out_file


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/PBS_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "PBS_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
PBS_rep2 = adata[adata.obs['name']=="PBS_rep2"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "PBS_rep2"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
PBS_rep2_plot = PBS_rep2.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = PBS_rep2_plot.obs["celltype4"].astype(str)

# Set `PBS_rep2_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
PBS_rep2_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        PBS_rep2_plot.obs["celltype4"].isna(),
        PBS_rep2_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
PBS_rep2_plot = PBS_rep2_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    PBS_rep2_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("PBS_rep2: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "PBS_rep2_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/WT_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "WT_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
WT_rep2 = adata[adata.obs['name']=="WT_rep2"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "WT_rep2"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
WT_rep2_plot = WT_rep2.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = WT_rep2_plot.obs["celltype4"].astype(str)

# Set `WT_rep2_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
WT_rep2_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        WT_rep2_plot.obs["celltype4"].isna(),
        WT_rep2_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
WT_rep2_plot = WT_rep2_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    WT_rep2_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("WT_rep2: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "WT_rep2_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep2_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep2"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
BE_rep2 = adata[adata.obs['name']=="BE_rep2"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "BE_rep2"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
BE_rep2_plot = BE_rep2.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = BE_rep2_plot.obs["celltype4"].astype(str)

# Set `BE_rep2_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
BE_rep2_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        BE_rep2_plot.obs["celltype4"].isna(),
        BE_rep2_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
BE_rep2_plot = BE_rep2_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep2_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("BE_rep2: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep2_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed SpatialData/Zarr object.
sdata = sd.read_zarr("Zarrfiles/Large_seg_with_morphology/BE_rep4_with_morphology.zarr")
sdata


In [ ]:
#spatial data uses pixels but xenium um
sample = "BE_rep4"
# 10x Xenium morphology pixel size
# In the 10x examples this is typically 0.2125 µm / pixel
um_per_px = 0.2125
# Calculate `px_per_um` from the existing values.
px_per_um = 1 / um_per_px


In [ ]:
# start again from the original polygons
cell_shapes = sdata.shapes["cell_boundaries"].copy()

# remove previous annotation columns if they exist
cols_to_drop = [
    "cell_id", "celltype4", "in_roi", 
    "celltype_x", "celltype_y", 
    "in_roi_x", "in_roi_y",
    "plot_color", "celltype_plot"
]

# Compute and store `cell_shapes`.
cell_shapes = cell_shapes.drop(
    columns=[c for c in cols_to_drop if c in cell_shapes.columns],
    errors="ignore"
)

# Make an independent copy of the selected data.
anno = (
    adata.obs
    .loc[adata.obs["name"] == sample, ["cell_id", "celltype4", "in_roi"]]
    .copy()
)

# Compute and store `cell_shapes_anno`.
cell_shapes_anno = cell_shapes.merge(
    anno,
    left_index=True,
    right_on="cell_id",
    how="left"
)

# Make an independent copy of the selected data.
cell_shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()


In [ ]:
# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# Define the values used for `(x0_um, x1_um)`.
x0_um, x1_um = roi["x_min"], roi["x_max"]
# Define the values used for `(y0_um, y1_um)`.
y0_um, y1_um = roi["y_min"], roi["y_max"]

# Calculate `x0_px` from the existing values.
x0_px = x0_um * px_per_um
# Calculate `x1_px` from the existing values.
x1_px = x1_um * px_per_um
# Calculate `y0_px` from the existing values.
y0_px = y0_um * px_per_um
# Calculate `y1_px` from the existing values.
y1_px = y1_um * px_per_um
# Run `print` for this analysis step.
print(x0_px, x1_px, y0_px, y1_px)


In [ ]:
# convert polygons from µm to pixels
cell_shapes_roi_px = cell_shapes_roi.copy()

# Compute and store `cell_shapes_roi_px['geometry']`.
cell_shapes_roi_px["geometry"] = cell_shapes_roi_px.geometry.scale(
    xfact=px_per_um,
    yfact=px_per_um,
    origin=(0, 0)
)


In [ ]:
# Run `print` for this analysis step.
print(cell_shapes.total_bounds)


In [ ]:
# Make an independent copy of the selected data.
BE_rep4 = adata[adata.obs['name']=="BE_rep4"].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]

# Define the values used for `highlight_ct`.
highlight_ct = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Set `sample` for the following analysis.
sample = "BE_rep4"

# ------------------------------------------------------------
# IMPORTANT:
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Make an independent copy of the selected data.
BE_rep4_plot = BE_rep4.copy()

# ------------------------------------------------------------
# Set all non-selected cell types to NA
# ------------------------------------------------------------

# Convert values to the required data type.
celltypes = BE_rep4_plot.obs["celltype4"].astype(str)

# Set `BE_rep4_plot.obs.loc[~celltypes.isin(selected_cts), 'celltype4']` for the following analysis.
BE_rep4_plot.obs.loc[
    ~celltypes.isin(selected_cts),
    "celltype4",
] = np.nan

# ------------------------------------------------------------
# Drawing order:
# grey cells first,
# selected populations second,
# highlighted populations last
#
# Highlighted populations are NOT plotted again,
# so their point size remains identical to all other cells.
# ------------------------------------------------------------

# Compute and store `draw_rank`.
draw_rank = np.select(
    [
        BE_rep4_plot.obs["celltype4"].isna(),
        BE_rep4_plot.obs["celltype4"].astype(str).isin(highlight_ct),
    ],
    [0, 2],
    default=1,
)

# Make an independent copy of the selected data.
BE_rep4_plot = BE_rep4_plot[np.argsort(draw_rank)].copy()

# ------------------------------------------------------------
# Create the figure explicitly so dimensions are fixed
# ------------------------------------------------------------

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Base spatial scatter plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep4_plot,
    color="celltype4",
    na_color="lightgrey",
    na_in_legend=False,
    spot_size=25,
    ax=ax,
    show=False,
)

# ------------------------------------------------------------
# ROI coordinates
# ------------------------------------------------------------

# Select the required subset and store it as `roi`.
roi = roi_squares_df.loc[sample]

# ROI and obsm["spatial"] are both in µm

# Select the required subset and store it as `x0`.
x0 = roi["x_min"]
# Select the required subset and store it as `x1`.
x1 = roi["x_max"]
# Select the required subset and store it as `y0`.
y0 = roi["y_min"]
# Select the required subset and store it as `y1`.
y1 = roi["y_max"]

# Run `print` for this analysis step.
print("ROI used for rectangle:", x0, x1, y0, y1)

# Compute and store `roi_rectangle`.
roi_rectangle = Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="black",
    facecolor="none",
    zorder=20,
)

# Run `ax.add_patch` for this analysis step.
ax.add_patch(roi_rectangle)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# These lines ensure that 1000 µm occupies the same physical
# distance in every exported sample plot.

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=10)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.3,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=20,
    fontproperties=fontprops,
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Title and axes
# ------------------------------------------------------------

# Add the plot title.
ax.set_title("BE_rep4: VCM populations with ROI")

# Remove visible coordinate axes

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Fixed margins help keep the exported canvas consistent

# Run `fig.subplots_adjust` for this analysis step.
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep4_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf",
    dpi=600,
)

# Display the completed figure.
plt.show()
